In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
TARGET = "price"
ID_COL = "id"

TAB_FEATURES = [
    "bedrooms","bathrooms","sqft_living","sqft_lot","floors","waterfront","view",
    "condition","grade","sqft_above","sqft_basement","yr_built","yr_renovated",
    "zipcode","lat","long","sqft_living15","sqft_lot15",
]


In [6]:
train_df = pd.read_csv("data/train(1).csv", dtype={"id": "string"})  # dtype supported by read_csv. [web:49]
test_df  = pd.read_csv("data/test2.csv",   dtype={"id": "string"})  # dtype supported by read_csv. [web:49]

train_df.shape, test_df.shape, train_df.columns.tolist()


((16209, 21),
 (5404, 20),
 ['id',
  'date',
  'price',
  'bedrooms',
  'bathrooms',
  'sqft_living',
  'sqft_lot',
  'floors',
  'waterfront',
  'view',
  'condition',
  'grade',
  'sqft_above',
  'sqft_basement',
  'yr_built',
  'yr_renovated',
  'zipcode',
  'lat',
  'long',
  'sqft_living15',
  'sqft_lot15'])

In [7]:
# If you want to use date later, parse it.
# pandas.read_csv supports parse_dates. [web:49]
train_df["date"] = pd.to_datetime(train_df["date"], errors="coerce")
test_df["date"]  = pd.to_datetime(test_df["date"], errors="coerce")

train_df[["date"]].head()


,date
0,2015-05-05
1,2014-07-08
2,2015-01-15
3,2015-04-27
4,2014-12-05


In [8]:
train_df["yr_renovated"] = train_df["yr_renovated"].fillna(0)
test_df["yr_renovated"]  = test_df["yr_renovated"].fillna(0)

medians = train_df[TAB_FEATURES].median(numeric_only=True)
train_df[TAB_FEATURES] = train_df[TAB_FEATURES].fillna(medians)
test_df[TAB_FEATURES]  = test_df[TAB_FEATURES].fillna(medians)

train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=RANDOM_SEED)

train_split.shape, val_split.shape


((12967, 21), (3242, 21))

In [9]:
scaler = StandardScaler()
scaler.fit(train_split[TAB_FEATURES])

X_train_tab = scaler.transform(train_split[TAB_FEATURES]).astype(np.float32)
X_val_tab   = scaler.transform(val_split[TAB_FEATURES]).astype(np.float32)
X_test_tab  = scaler.transform(test_df[TAB_FEATURES]).astype(np.float32)

y_train = train_split[TARGET].astype(np.float32).values
y_val   = val_split[TARGET].astype(np.float32).values

train_ids = train_split[ID_COL].astype("string").values
val_ids   = val_split[ID_COL].astype("string").values
test_ids  = test_df[ID_COL].astype("string").values


In [10]:
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

np.save(out_dir / "X_train_tab.npy", X_train_tab)
np.save(out_dir / "X_val_tab.npy",   X_val_tab)
np.save(out_dir / "X_test_tab.npy",  X_test_tab)

np.save(out_dir / "y_train.npy", y_train)
np.save(out_dir / "y_val.npy",   y_val)

pd.DataFrame({"id": train_ids}).to_csv(out_dir / "train_ids.csv", index=False)
pd.DataFrame({"id": val_ids}).to_csv(out_dir / "val_ids.csv", index=False)
pd.DataFrame({"id": test_ids}).to_csv(out_dir / "test_ids.csv", index=False)

pd.DataFrame({"feature": TAB_FEATURES, "mean": scaler.mean_, "scale": scaler.scale_}).to_csv(
    out_dir / "tabular_scaler.csv", index=False
)

out_dir


PosixPath('data/processed')